In [ ]:
# !pip install google-api-python-client pandas openpyxl -q

In [ ]:
import os

import re
import time
import pandas as pd
from googleapiclient.discovery import build


# =========================
# 1. API 키 입력
# =========================
API_KEY = os.environ["YOUTUBE_API_KEY"]

youtube = build("youtube", "v3", developerKey=API_KEY)


# =========================
# 2. 수집할 유튜브 영상 URL 입력
# =========================
video_urls = [
    "https://www.youtube.com/watch?v=gmDLg2hqPNY",
    "https://www.youtube.com/watch?v=tKxsHwDKErQ",
    "https://www.youtube.com/watch?v=TqobxVKgrw4",
    "https://www.youtube.com/watch?v=ngJ8zrvDyrM",
    "https://www.youtube.com/watch?v=OLE8onRDxtI",
    "https://www.youtube.com/watch?v=6Plq5IO9Q7I",
    "https://www.youtube.com/watch?v=QxmZlyiOHmA",
    "https://www.youtube.com/watch?v=61VqM99jKfQ",
    "https://www.youtube.com/watch?v=xLOviW8nML0",
    "https://www.youtube.com/watch?v=q-Zut_8mp1k",
    "https://www.youtube.com/watch?v=pxOYNRudATU"

]
match_contexts = [
    "프리미어리그 38라운드: 아스날 2:1 승리. 아스날과 상대팀 모두 득점한 경기.",
    "프리미어리그 37라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 36라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 35라운드: 아스날 3:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 34라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 33라운드: 아스날 1:2 패배. 아스날이 패배한 경기.",
    "프리미어리그 32라운드: 아스날 1:2 패배. 아스날이 패배한 경기.",
    "프리미어리그 30라운드: 아스날 2:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 29라운드: 아스날 1:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 28라운드: 아스날 2:1 승리. 아스날과 상대팀 모두 득점한 경기.",
    "UEFA 챔피언스리그 결승: 아스날 1:1 무승부 후 승부차기 패배. 아스날이 준우승한 경기이며, 조롱·비꼼·아쉬움이 포함될 수 있는 영상."
]


# =========================
# 3. 영상 ID 추출 함수
# =========================
def get_video_id(url):
    patterns = [
        r"v=([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"shorts/([^?&]+)",
        r"embed/([^?&]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    return None


# =========================
# 4. 댓글 최소 전처리 함수
# =========================
def clean_comment(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # HTML 태그 제거
    text = re.sub(r"<.*?>", " ", text)

    # URL 제거
    text = re.sub(r"http\S+|www\S+", " ", text)

    # 줄바꿈 제거
    text = text.replace("\n", " ").replace("\r", " ")

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================
# 5. 댓글 수집
# =========================
all_comments = []
seen_comment_ids = set()

# 비정상 무한 수집 방지용 안전장치
MAX_COMMENTS_PER_VIDEO = 5000

print("수집할 영상 개수:", len(video_urls))
print("=" * 60)

for idx, (url, match_context) in enumerate(zip(video_urls, match_contexts), start=1):
    video_id = get_video_id(url)

    if video_id is None:
        print(f"[스킵] 영상 ID 추출 실패: {url}")
        continue

    print(f"\n[{idx}/{len(video_urls)}] 댓글 수집 시작: {video_id}")

    page_count = 0
    video_comment_count = 0
    next_page_token = None

    try:
        while True:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                textFormat="plainText",
                order="relevance",
                pageToken=next_page_token
            )

            response = request.execute()
            page_count += 1

            items = response.get("items", [])
            new_count_this_page = 0

            for item in items:
                comment = item["snippet"]["topLevelComment"]
                comment_id = comment["id"]

                # 이미 수집한 댓글이면 스킵
                if comment_id in seen_comment_ids:
                    continue

                seen_comment_ids.add(comment_id)

                comment_data = comment["snippet"]

                original_comment = comment_data.get("textDisplay", "")
                clean_text = clean_comment(original_comment)

                all_comments.append({
                    "comment_id": comment_id,
                    "video_id": video_id,
                    "video_url": url,
                    "match_context": match_context,
                    "author": comment_data.get("authorDisplayName"),
                    "comment_original": original_comment,
                    "comment_clean": clean_text,
                    "like_count": comment_data.get("likeCount"),
                    "published_at": comment_data.get("publishedAt")
                })

                video_comment_count += 1
                new_count_this_page += 1

            print(
                f"[진행중] 영상 {video_id} | "
                f"{page_count}페이지 완료 | "
                f"이번 페이지 신규 {new_count_this_page}개 | "
                f"현재 {video_comment_count}개 수집"
            )

            # 이번 페이지에서 새 댓글이 하나도 없으면 반복 오류 가능성 있음
            if new_count_this_page == 0:
                print(f"[중단] 영상 {video_id} | 새 댓글이 0개라 반복 수집 방지 중단")
                break

            # 안전장치: 말도 안 되게 많이 수집되면 중단
            if video_comment_count >= MAX_COMMENTS_PER_VIDEO:
                print(f"[중단] 영상 {video_id} | {MAX_COMMENTS_PER_VIDEO}개 도달, 비정상 반복 방지")
                break

            next_page_token = response.get("nextPageToken")

            # 다음 페이지 없으면 정상 종료
            if not next_page_token:
                break

            time.sleep(0.1)

        print(f"[완료] 영상 {video_id} | 총 {video_comment_count}개 수집")

    except Exception as e:
        print(f"[에러 발생] 영상 {video_id}")
        print(e)
        continue


# =========================
# 6. 데이터프레임 생성 및 저장
# =========================
df = pd.DataFrame(all_comments)

print("\n" + "=" * 60)
print("전체 원본 수집 댓글 수:", len(df))

if len(df) > 0:
    df = df[df["comment_clean"].str.len() > 0].copy()

    # 혹시 모를 중복 최종 제거
    df = df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)

    print("중복 제거 후 댓글 수:", len(df))

    df.to_csv(
        "youtube_comments_clean2.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df.to_excel(
        "youtube_comments_clean2.xlsx",
        index=False
    )

    print("저장 완료")
    print("CSV 파일: youtube_comments_clean2.csv")
    print("Excel 파일: youtube_comments_clean2.xlsx")

else:
    print("수집된 댓글이 없습니다.")


# =========================
# 7. 확인
# =========================
df.head()

# GPT API 1차 감정분석

In [ ]:
# !pip install openai tqdm

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from openai import OpenAI

client = OpenAI()

# =========================
# 1. 댓글 파일 불러오기
# =========================
df = pd.read_csv("youtube_comments_clean2.csv")

print("전체 댓글 수:", len(df))
df.head()

In [ ]:
# =========================
# 2. 감정분석 함수
# =========================
def analyze_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 한국어 축구 팬덤 댓글을 분석하는 target-aware 감정분석 분류기다.
분석 대상은 EPL 아스날 관련 유튜브 댓글이다.

[아스날 선수단 정보]
감독: 미켈 아르테타

선수:
가브리엘 마갈량이스, 가브리엘 마르티넬리, 가브리엘 제주스,
노니 마두에케, 다비드 라야, 데클란 라이스, 레안드로 트로사르,
리카르도 칼라피오리, 마르틴 수비멘디, 마르틴 외데고르,
마일스 루이스-스켈리, 막스 다우만, 말리 샐먼, 미켈 메리노,
벤 화이트, 부카요 사카, 빅토르 요케레스, 안드레 해리먼-아누스,
에베레치 에제, 위리엔 팀버르, 윌리엄 살리바, 이선 은와네리,
카이 하베르츠, 케파 아리사발라가, 크리스티안 뇌가르,
크리스티안 모스케라, 토미 셋포드, 피에로 인카피에, 야쿠브 키비오르

[선수단 반영 규칙]
- 위 선수나 감독을 칭찬하면 아스날 관련 positive로 분류한다.
- 위 선수나 감독을 비판, 조롱, 실망하면 아스날 관련 negative로 분류한다.
- 위 선수 이름이 직접 나오면 target은 보통 arsenal_player로 둔다.
- 미켈 아르테타 관련 댓글이면 target은 arsenal_manager로 둔다.
- 이강인만 중심으로 언급한 댓글은 아스날 감정으로 보지 않고 neutral로 분류한다.
- 이강인 칭찬, 이강인 이적 희망, 이강인 관련 단순 언급은 neutral로 처리한다.
- 단, 이강인 댓글 안에서도 아스날 선수나 아스날 팀을 직접 칭찬/비판하면 그 부분을 기준으로 판단한다.


[영상/경기 맥락]
{match_context}

이 맥락은 댓글 해석에 반드시 반영한다.
아스날이 승리한 경기에서는 "골 미쳤다", "마무리 좋다", "슈팅 지렸다", "수비 좋다"처럼 대상이 불명확한 경기력 칭찬은 특별히 상대팀 언급이 없으면 아스날 positive로 우선 판단한다.
아스날이 패배한 경기에서는 아쉬움, 비판, 조롱, 실망 표현이 아스날을 향하면 negative로 판단한다.
챔피언스리그 결승 패배 영상에서는 "역시 아스날", "준우승 DNA", "또 2등", "아스날답다", "ㅋㅋㅋㅋ" 등이 비꼼/조롱이면 negative로 판단한다.

[분류 기준]
- positive: 아스날 팀/선수/감독/전술/경기력/결과에 대한 칭찬, 감탄, 기대, 축하
- negative: 아스날 팀/선수/감독/전술/경기력/결과에 대한 비판, 실망, 조롱, 비꼼, 분노
- neutral: 아스날과 직접 관련 없거나 단순 질문/정보/감정 판단 불가

[중요 규칙]
- neutral은 정말 감정 방향을 판단할 수 없을 때만 사용한다.
- 아스날 관련 맥락이 있고 긍정/부정 뉘앙스가 조금이라도 있으면 positive 또는 negative로 판단한다.
- "미쳤다", "지린다", "개쩐다", "폼 미쳤다", "찢었다", "월클", "GOAT"는 칭찬 맥락이면 positive.
- "개못한다", "답답하다", "망했다", "처참하다", "노답", "왜 저러냐"는 비판 맥락이면 negative.
- 상대팀/상대선수 칭찬은 아스날과 무관하면 neutral.
- 단, 상대팀 칭찬이 아스날의 패배/부진/약점을 강조하면 negative.
-- 단순 감탄사라도 영상 맥락상 아스날 경기 장면에 대한 반응으로 해석 가능하면 neutral보다 positive/negative를 우선한다.

[target 기준]
- arsenal_team
- arsenal_player
- arsenal_manager
- opponent
- referee_or_league
- lee_kangin
- unclear

[스코어/라운드 맥락 활용]
라운드 정보와 스코어는 댓글 해석에 반영한다.
match_context에서 상대팀이 득점하지 못한 경기라고 주어진 경우,
"골 미쳤다", "마무리 좋다", "슈팅 지렸다", "득점 장면 좋다"처럼 득점/공격 장면을 칭찬하는 댓글은
특별히 상대팀이나 상대 선수가 명시되지 않는 한 아스날 positive로 판단한다.

아스날이 패배한 경기에서는 경기력 비판, 실망, 조롱, 비꼼 표현을 negative로 판단한다.
단, 패배 경기라도 아스날 선수 개인의 좋은 플레이를 칭찬하면 positive로 판단한다.

챔피언스리그 결승 맥락에서는 "역시 아스날", "준우승 DNA", "또 2등", "아스날답다", "ㅋㅋㅋㅋ" 등이
비꼼/조롱이면 negative로 판단한다.

[출력 규칙]
반드시 JSON 배열만 출력하라.
설명 문장, 코드블록, 마크다운은 쓰지 마라.
각 결과에는 입력 번호 id를 포함하라.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0부터 1.0 사이 숫자,
    "target": "arsenal_team 또는 arsenal_player 또는 arsenal_manager 또는 opponent 또는 referee_or_league 또는 lee_kangin 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[분석할 댓글]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "너는 한국어 축구 댓글 감정분석 전문가다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [{"id": i, "sentiment": "error", "confidence": 0.0, "target": "error", "reason": str(e)} for i in range(len(comments))]

In [ ]:
# =========================
# 전체 댓글 실행 + 1차 neutral 댓글 2차 재분석 + 최종 저장
# =========================

import re
import json
import time
import pandas as pd
from tqdm import tqdm


# =========================
# 1. GPT 응답 JSON 파싱 보정 함수
# =========================
# 모델이 가끔 ```json 같은 코드블록을 붙이거나,
# 앞뒤에 설명 문장을 붙이는 경우가 있어서 JSON 배열만 잘라내는 함수
def parse_json_array(text):
    text = text.strip()

    # 코드블록 제거
    text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()

    # JSON 배열 부분만 추출
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1:
        text = text[start:end+1]

    return json.loads(text)


# =========================
# 2. 1차 전체 댓글 감정분석 실행 함수
# =========================
# 기존 analyze_batch() 함수를 사용해서 전체 댓글을 먼저 positive / negative / neutral로 분류
def run_batch_labeling(df, batch_size=20):
    all_results = []

    for video_id, video_df in tqdm(df.groupby("video_id")):
        video_df = video_df.reset_index(drop=False)
        match_context = video_df["match_context"].iloc[0]

        for start in range(0, len(video_df), batch_size):
            batch_df = video_df.iloc[start:start+batch_size]
            comments = batch_df["comment_clean"].tolist()

            batch_results = analyze_batch(comments, match_context)
            result_map = {r.get("id"): r for r in batch_results}

            for i, row in batch_df.iterrows():
                local_id = i - start
                r = result_map.get(local_id, {})

                all_results.append({
                    "original_index": row["index"],
                    "gpt_sentiment": r.get("sentiment", "error"),
                    "gpt_confidence": r.get("confidence", 0.0),
                    "gpt_target": r.get("target", "unclear"),
                    "gpt_reason": r.get("reason", "")
                })

            time.sleep(0.3)

    result_df = pd.DataFrame(all_results)
    result_df = result_df.sort_values("original_index").reset_index(drop=True)

    final_df = df.reset_index(drop=True).copy()
    final_df["gpt_sentiment"] = result_df["gpt_sentiment"]
    final_df["gpt_confidence"] = result_df["gpt_confidence"]
    final_df["gpt_target"] = result_df["gpt_target"]
    final_df["gpt_reason"] = result_df["gpt_reason"]

    return final_df


# =========================
# 3. 2차 neutral 재분석 프롬프트 함수
# =========================
# 1차에서 neutral로 나온 댓글만 다시 검사해서,
# 조금이라도 긍정/부정 뉘앙스가 있으면 positive 또는 negative로 바꾸는 함수
def analyze_neutral_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 아스날 관련 유튜브 댓글의 2차 감정분석 분류기다.
아래 댓글들은 1차 분석에서 neutral로 분류되었지만, neutral 판정을 최소화하기 위해 재검토한다.

[경기 맥락]
{match_context}

[가장 중요한 원칙]
neutral은 예외값이다.
감정 방향이 정말 전혀 없을 때만 neutral로 둔다.

[이강인 처리 예외 규칙]
단, 이강인 중심 댓글은 예외적으로 neutral을 유지한다.
이강인 칭찬, 이강인 이적 희망, 이강인 관련 단순 언급은 아스날 감정으로 보지 않는다.
예: "이강인 아스날 가면 좋겠다", "강인이 패스 좋네", "이강인 잘한다","대강인" → neutral, target lee_kangin
이강인 댓글 안에서도 아스날 팀, 아스날 선수, 아스날 감독을 직접 칭찬하거나 비판하면 그 부분을 기준으로 positive 또는 negative를 판단한다.
예: "이강인보다 외데고르가 낫다" → positive, target arsenal_player
예: "이강인이 외데고르보다 낫다" → negative, target arsenal_player

칭찬, 감탄, 만족, 축하, 기대가 조금이라도 있으면 positive.
비판, 실망, 조롱, 분노, 답답함, 비꼼이 조금이라도 있으면 negative.

댓글이 짧거나 대상이 생략되어도,
경기 맥락상 아스날에 대한 반응으로 자연스럽게 해석되면
positive 또는 negative로 분류한다.

[neutral로 유지하는 경우]
- 단순 질문
- 경기 시간, 일정, 정보 확인
- 영상 편집, 중계, 해설, 업로드 관련 이야기
- 아스날 / 아스날 선수 / 감독 / 경기 결과와 감정적으로 연결되지 않는 말
- 감정 단서가 전혀 없는 단순 사실 전달

[neutral 금지 예시]
"미쳤다", "지렸다", "좋다", "깔끔하다", "나이스", "레전드", "GOAT" → positive
"답답하다", "망했다", "왜 저러냐", "최악", "또 이러네", "아쉽다" → negative

"역시 아스날"은 칭찬이면 positive, 조롱/비꼼이면 negative.
상대팀 칭찬이라도 아스날의 패배, 약점, 부진을 강조하면 negative.

[판단 우선순위]
1. 비판 / 조롱 / 실망 단서가 있으면 negative
2. 칭찬 / 감탄 / 만족 단서가 있으면 positive
3. 둘 다 전혀 없을 때만 neutral

단, 이강인 중심 댓글은 예외로 neutral을 유지한다.
그 외 아스날 관련 댓글이 애매하면 neutral 대신 positive 또는 negative 중 더 가까운 쪽을 선택한다.

[출력 규칙]
반드시 JSON 배열만 출력한다.
설명 문장, 코드블록, 마크다운은 출력하지 않는다.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0,
    "target": "arsenal_team 또는 arsenal_player 또는 arsenal_manager 또는 opponent 또는 referee_or_league 또는 lee_kangin 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[재분석할 댓글]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 축구 댓글 감정분석 전문가다. neutral 판정은 매우 보수적으로 사용한다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "neutral",
                "confidence": 0.0,
                "target": "unclear",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]


# =========================
# 4. 1차 neutral 댓글만 골라서 2차 재분석 실행
# =========================
def rerun_neutral_second_pass(labeled_df, batch_size=20):
    final_df = labeled_df.reset_index(drop=True).copy()

    # 1차 결과 보존용 컬럼
    final_df["gpt_sentiment_1st"] = final_df["gpt_sentiment"]
    final_df["gpt_reason_1st"] = final_df["gpt_reason"]

    # 어떤 댓글이 2차 재분석 대상이었는지 표시
    final_df["neutral_second_pass"] = False

    neutral_df = final_df[
        final_df["gpt_sentiment"].astype(str).str.lower().eq("neutral")
    ].copy()

    print("2차 재분석 대상 neutral 댓글 수:", len(neutral_df))

    for video_id, video_df in tqdm(neutral_df.groupby("video_id")):
        match_context = video_df["match_context"].iloc[0]

        for start in range(0, len(video_df), batch_size):
            batch_df = video_df.iloc[start:start+batch_size]
            comments = batch_df["comment_clean"].tolist()

            batch_results = analyze_neutral_batch(comments, match_context)

            result_map = {}
            for r in batch_results:
                try:
                    result_map[int(r.get("id"))] = r
                except:
                    pass

            for local_id, row_index in enumerate(batch_df.index):
                r = result_map.get(local_id, {})

                new_sentiment = str(
                    r.get("sentiment", "neutral")
                ).lower().strip()

                if new_sentiment not in ["positive", "negative", "neutral"]:
                    new_sentiment = "neutral"

                final_df.at[row_index, "gpt_sentiment"] = new_sentiment
                final_df.at[row_index, "gpt_confidence"] = r.get("confidence", 0.0)
                final_df.at[row_index, "gpt_target"] = r.get("target", "unclear")
                final_df.at[row_index, "gpt_reason"] = "[2차 neutral 재분석] " + str(r.get("reason", ""))
                final_df.at[row_index, "neutral_second_pass"] = True

            time.sleep(0.3)

    return final_df


# =========================
# 5. 댓글 파일 불러오기
# =========================
df = pd.read_csv("youtube_comments_clean2.csv")

print("전체 댓글 수:", len(df))


# =========================
# 6. 1차 전체 감정분석 실행
# =========================
arsenal_labeled_df = run_batch_labeling(df, batch_size=20)

print("1차 분석 결과")
print(arsenal_labeled_df["gpt_sentiment"].value_counts())


# =========================
# 7. 1차 neutral 댓글만 2차 재분석 실행
# =========================
arsenal_labeled_df = rerun_neutral_second_pass(
    arsenal_labeled_df,
    batch_size=20
)

print("2차 재분석 후 결과")
print(arsenal_labeled_df["gpt_sentiment"].value_counts())


# =========================
# 이강인 중심 댓글은 아스날 감정분석 대상이 아니므로 neutral 보정
# =========================
lee_keywords = r"이강인|강인|대강인|kangin|kang-in|lee kang|lee kang-in"

lee_mask = arsenal_labeled_df["comment_clean"].astype(str).str.contains(
    lee_keywords,
    case=False,
    na=False,
    regex=True
)

print("이강인 관련 neutral 보정 댓글 수:", lee_mask.sum())

arsenal_labeled_df.loc[lee_mask, "gpt_sentiment"] = "neutral"
arsenal_labeled_df.loc[lee_mask, "gpt_target"] = "lee_kangin"
arsenal_labeled_df.loc[lee_mask, "gpt_confidence"] = 1.0
arsenal_labeled_df.loc[lee_mask, "gpt_reason"] = (
    "[규칙 기반 neutral 보정] 이강인 중심 댓글은 아스날 팀/선수/감독에 대한 감정분석 대상이 아니므로 neutral 처리."
)

print("이강인 보정 후 최종 결과")
print(arsenal_labeled_df["gpt_sentiment"].value_counts())


# =========================
# 8. 최종 파일 저장
# =========================
# 기존 1차 파일과 헷갈리지 않게 neutral_refined 이름으로 저장
arsenal_labeled_df.to_excel(
    "arsenal_gpt_all_labeled_batch_neutral_refined.xlsx",
    index=False
)

arsenal_labeled_df.to_csv(
    "arsenal_gpt_all_labeled_batch_neutral_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("Excel 파일: arsenal_gpt_all_labeled_batch_neutral_refined.xlsx")
print("CSV 파일: arsenal_gpt_all_labeled_batch_neutral_refined.csv")


# =========================
# 9. 결과 미리보기
# =========================
arsenal_labeled_df[[
    "comment_clean",
    "match_context",
    "gpt_sentiment_1st",
    "gpt_sentiment",
    "gpt_confidence",
    "gpt_target",
    "gpt_reason",
    "neutral_second_pass"
]].head()